# **Import Libraries**

In [ ]:
import os
import copy
import random
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold

In [ ]:
from utils.experiment_separator import (
    separate_experiments,
    get_experiment_summary,
    print_experiment_report,
    get_experiments,
)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# **Import Dataset**

In [6]:
PATH = "../data/data.csv"
df_raw = pd.read_csv(PATH)

# **Rename Columns**

In [7]:
df = df_raw.copy()
df.columns = ['density', 'cutting_speed', 'feed_rate', 'depth', 'axial_force', 'cutting_force']
TARGET   = 'cutting_force'
FEATURES = ['density', 'cutting_speed', 'feed_rate', 'depth']
INPUT = ['density', 'cutting_speed', 'feed_rate', 'depth', 'axial_force']

# **Separate Experiments**

In [8]:
df = separate_experiments(df)
print_experiment_report(df)
experiments = get_experiments(df)

Total unique combinations : 72
Experiment count range    : 2 – 3
Total experiments         : 198

── Experiments per combination ──
 density  cutting_speed  feed_rate  experiment_count
      10             10         10                 3
      10             10         15                 3
      10             10         20                 3
      10             16         10                 3
      10             16         15                 3
      10             16         20                 3
      10             25         10                 3
      10             25         15                 3
      10             25         20                 3
      10             40         10                 3
      10             40         15                 3
      10             40         20                 3
      10             63         10                 3
      10             63         15                 3
      10             63         20                 3
      10            

# **Preprocessing Pipeline**

## Data Cleaning

In [9]:
def apply_remove_null(df: pd.DataFrame) -> pd.DataFrame:
    return df.dropna()

## Data Scaling

In [10]:
def apply_scaling(df: pd.DataFrame, scaler: MinMaxScaler, fit: bool = False) -> pd.DataFrame:
    cols = INPUT + [TARGET]
    if fit:
        df[cols] = scaler.fit_transform(df[cols])
    else:
        df[cols] = scaler.transform(df[cols])
    return df

## Feature Engineering

In [11]:
def apply_add_feature(df: pd.DataFrame) -> pd.DataFrame:
    return df

## Data Pipeline

In [ ]:
class Pipeline:
    def __init__(self):
        self.scaler = MinMaxScaler()
 
    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        temp = df.copy()
        temp = apply_remove_null(temp)
        temp = apply_scaling(temp, self.scaler, fit=True)
        temp = apply_add_feature(temp)
        return temp
 
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        temp = df.copy()
        temp = apply_remove_null(temp)
        temp = apply_scaling(temp, self.scaler, fit=False)
        temp = apply_add_feature(temp)
        return temp

# **Train/Val Split**

In [ ]:
exp_keys = list(experiments.keys())
random.shuffle(exp_keys)
 
n_total      = len(exp_keys)
n_train_pool = int(n_total * 0.85)
 
train_pool_keys = exp_keys[:n_train_pool]
test_keys       = exp_keys[n_train_pool:]
 
print(f"\nTotal experiments      : {n_total}")
print(f"Train pool experiments : {len(train_pool_keys)}")
print(f"Test experiments       : {len(test_keys)}")
 
N_FOLDS = 3
if len(train_pool_keys) < N_FOLDS:
    raise ValueError(f"train_pool_keys {len(train_pool_keys)} experiment, kurang dari N_FOLDS={N_FOLDS}")


Total experiments : 198
Train experiments : 138
Val experiments   : 29   <- dipakai buat checkpointing pas training
Test experiments  : 31  <- baru disentuh sekali, pas evaluasi akhir


## Windowing

In [ ]:
def experiments_to_tensor(keys, experiments, pipeline, window_size, fit=False):
    ref_df = pd.concat([experiments[k] for k in keys], ignore_index=True)
    processed = pipeline.fit_transform(ref_df) if fit else pipeline.transform(ref_df)
 
    lengths = [len(experiments[k]) for k in keys]
 
    X_windows, y_windows = [], []
    start = 0
    for length in lengths:
        exp_slice = processed.iloc[start:start+length].reset_index(drop=True)
        start += length
 
        if len(exp_slice) < window_size:
            continue
 
        input_vals  = exp_slice[INPUT].values
        target_vals = exp_slice[TARGET].values
 
        for i in range(len(exp_slice) - window_size + 1):
            X_windows.append(input_vals[i:i+window_size])
            y_windows.append(target_vals[i+window_size-1])
 
    X = torch.tensor(np.array(X_windows), dtype=torch.float32)
    y = torch.tensor(np.array(y_windows), dtype=torch.float32).unsqueeze(1)
    return X, y

# **Modeling**

In [ ]:
INPUT_SIZE  = len(INPUT)
OUTPUT_SIZE = 1

## hyperparameter

In [ ]:
WINDOW_SIZE   = 10
HIDDEN_SIZE   = 64
NUM_LAYERS    = 2
EPOCHS        = 10
BATCH_SIZE    = 32
LEARNING_RATE = 1e-3

## Training Loop

In [15]:
def train(model_class, model_name: str, experiments: dict, train_keys: list, val_keys: list):
    kf             = KFold(n_splits=N_FOLDS, shuffle=False)
    train_key_arr  = np.array(train_keys, dtype=object)
    fold_val_losses = []
 
    print(f"\n Training {model_name} ")
 
    for fold, (fold_train_idx, fold_val_idx) in enumerate(kf.split(train_key_arr)):
        fold_train_keys = [tuple(k) for k in train_key_arr[fold_train_idx]]
        fold_val_keys   = [tuple(k) for k in train_key_arr[fold_val_idx]]
        
        model     = model_class()
        pipeline  = Pipeline()
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=model.lr)
 
        X_fold_train, y_fold_train = experiments_to_tensor(fold_train_keys, experiments, pipeline, fit=True)
        X_fold_val,   y_fold_val   = experiments_to_tensor(fold_val_keys,   experiments, pipeline, fit=False)
 
        train_loader = DataLoader(TensorDataset(X_fold_train, y_fold_train), batch_size=model.batch_size, shuffle=True)
 
        for epoch in range(model.epochs):
            model.train()
            train_loss = 0
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                loss = criterion(model(X_batch), y_batch)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
 
            model.eval()
            with torch.no_grad():
                val_loss = criterion(model(X_fold_val), y_fold_val).item()
 
            if True:
                print(f"  Fold {fold+1}/{N_FOLDS} | Epoch {epoch+1}/{model.epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss:.4f}")
 
        fold_val_losses.append(val_loss)
 
    print(f"\n  KFold Val Loss per fold (diagnostic only) : {[f'{v:.4f}' for v in fold_val_losses]}")
    print(f"  Mean Val Loss                             : {np.mean(fold_val_losses):.4f} ± {np.std(fold_val_losses):.4f}")
    print(f"\n  Retraining on full train set, checkpointing on val_keys...")
    final_model    = model_class()
    final_pipeline = Pipeline()
    criterion      = nn.MSELoss()
    optimizer      = torch.optim.Adam(final_model.parameters(), lr=final_model.lr)
 
    X_train, y_train = experiments_to_tensor(train_keys, experiments, final_pipeline, fit=True)
    X_val,   y_val    = experiments_to_tensor(val_keys,   experiments, final_pipeline, fit=False)
    train_loader     = DataLoader(TensorDataset(X_train, y_train), batch_size=final_model.batch_size, shuffle=True)
 
    best_val_loss   = float('inf')
    best_state_dict = None
    best_epoch      = -1
 
    for epoch in range(final_model.epochs):
        final_model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(final_model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
 
        final_model.eval()
        with torch.no_grad():
            val_loss = criterion(final_model(X_val), y_val).item()
 
        if val_loss < best_val_loss:
            best_val_loss   = val_loss
            best_state_dict = copy.deepcopy(final_model.state_dict())
            best_epoch      = epoch + 1
 
    print(f"  Best epoch on val_keys: {best_epoch} | Best Val Loss: {best_val_loss:.4f}")
    final_model.load_state_dict(best_state_dict)
 
    return final_model, final_pipeline

## Validation Plot

In [16]:
def inverse_transform_target(pipeline, values):
    n_cols = len(INPUT) + 1
    dummy = np.zeros((len(values), n_cols))
    dummy[:, -1] = values
    return pipeline.scaler.inverse_transform(dummy)[:, -1]
 
 
def compute_experiment_predictions(model, key, experiments, pipeline):
    exp_df = experiments[key].copy()
    processed = pipeline.transform(exp_df)
 
    if len(processed) < WINDOW_SIZE:
        return None 
    input_vals  = processed[INPUT].values
    target_vals = processed[TARGET].values
    depth_vals  = exp_df['depth'].values
    X_windows, y_windows, depth_windows = [], [], []
    for i in range(len(processed) - WINDOW_SIZE + 1):
        X_windows.append(input_vals[i:i+WINDOW_SIZE])
        y_windows.append(target_vals[i+WINDOW_SIZE-1])
        depth_windows.append(depth_vals[i+WINDOW_SIZE-1])
    X = torch.tensor(np.array(X_windows), dtype=torch.float32)
    y = torch.tensor(np.array(y_windows), dtype=torch.float32).unsqueeze(1)
 
    model.eval()
    with torch.no_grad():
        preds = model(X).squeeze().numpy()
    actuals = y.squeeze().numpy()
 
    preds_orig   = inverse_transform_target(pipeline, preds)
    actuals_orig = inverse_transform_target(pipeline, actuals)
    depth_orig   = np.array(depth_windows)
 
    return depth_orig, actuals_orig, preds_orig

## Validation Plot for Each Model

In [17]:
def plot_validation_overview(model, model_name, val_keys, experiments, pipeline, output_root="validation_plots"):
    out_dir = os.path.join(output_root, model_name)
    os.makedirs(out_dir, exist_ok=True)
 
    exp_numbers, mean_actuals, mean_preds = [], [], []
    all_actuals, all_preds = [], []
 
    for idx, key in enumerate(val_keys, start=1):
        result = compute_experiment_predictions(model, key, experiments, pipeline)
        if result is None:
            continue
        _, actuals_orig, preds_orig = result
        exp_numbers.append(idx)
        mean_actuals.append(np.mean(actuals_orig))
        mean_preds.append(np.mean(preds_orig))
        all_actuals.extend(actuals_orig)
        all_preds.extend(preds_orig)
 
    rmse_of_means   = np.sqrt(mean_squared_error(mean_actuals, mean_preds))
    rmse_per_window = np.sqrt(mean_squared_error(all_actuals, all_preds))
 
    plt.figure(figsize=(10, 4))
    plt.plot(exp_numbers, mean_actuals, label="Actual",    color="steelblue", marker='o')
    plt.plot(exp_numbers, mean_preds,   label="Predicted", color="tomato", linestyle="--", marker='x')
    plt.title(f"{model_name} — Validation Overview (per-window RMSE: {rmse_per_window:.4f})")
    plt.xlabel("Experiment Number")
    plt.ylabel("cutting_force (mean, unscaled)")
    plt.xticks(exp_numbers)
    plt.legend()
    plt.tight_layout()
 
    save_path = os.path.join(out_dir, f"{model_name}_validation_overview.png")
    plt.savefig(save_path)
    plt.close()
    print(f"  Saved: {save_path}  |  RMSE of per-exp means: {rmse_of_means:.4f}  |  True per-window RMSE: {rmse_per_window:.4f}")

## Validation Plot for Each Experiment

In [18]:
def plot_and_save_per_experiment(model, model_name, val_keys, experiments, pipeline, output_root="validation_plots"):
    out_dir = os.path.join(output_root, model_name)
    os.makedirs(out_dir, exist_ok=True)
 
    for key in val_keys:
        result = compute_experiment_predictions(model, key, experiments, pipeline)
        if result is None:
            continue
        depth_orig, actuals_orig, preds_orig = result
 
        sort_idx       = np.argsort(depth_orig)
        depth_sorted   = depth_orig[sort_idx]
        actuals_sorted = actuals_orig[sort_idx]
        preds_sorted   = preds_orig[sort_idx]
 
        rmse_exp = np.sqrt(mean_squared_error(actuals_orig, preds_orig))
 
        exp_name  = "_".join(str(k) for k in key) if isinstance(key, tuple) else str(key)
        safe_name = exp_name.replace(" ", "").replace("/", "-")
 
        plt.figure(figsize=(10, 4))
        plt.plot(depth_sorted, actuals_sorted, label="Actual", color="steelblue", marker='o')
        plt.plot(depth_sorted, preds_sorted,   label="Predicted", color="tomato", linestyle="--", marker='x')
        plt.title(f"{model_name} RMSE: {rmse_exp:.4f} (Experiment {exp_name})")
        plt.xlabel("depth")
        plt.ylabel("cutting_force")
        plt.legend()
        plt.tight_layout()
 
        save_path = os.path.join(out_dir, f"{safe_name}.png")
        plt.savefig(save_path)
        plt.close()
        print(f"  Saved: {save_path}  |  RMSE: {rmse_exp:.4f}")

## LSTM

In [19]:
class LSTM(nn.Module):
    hidden_size = 64
    num_layers  = 2
    epochs      = 10
    batch_size  = 32
    lr          = 1e-3
 
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(INPUT_SIZE, self.hidden_size, self.num_layers, batch_first=True)
        self.fc   = nn.Linear(self.hidden_size, OUTPUT_SIZE)
 
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

In [20]:
lstm_model, lstm_pipeline = train(LSTM, "LSTM", experiments, train_keys, val_keys)
X_test_lstm, y_test_lstm    = experiments_to_tensor(test_keys, experiments, lstm_pipeline, fit=False)
 
plot_validation_overview(lstm_model, "LSTM", test_keys, experiments, lstm_pipeline)
plot_and_save_per_experiment(lstm_model, "LSTM", test_keys, experiments, lstm_pipeline)


 Training LSTM 
  Fold 1/3 | Epoch 1/10 | Train Loss: 0.0004 | Val Loss: 0.0001
  Fold 1/3 | Epoch 2/10 | Train Loss: 0.0001 | Val Loss: 0.0000
  Fold 1/3 | Epoch 3/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 1/3 | Epoch 4/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 1/3 | Epoch 5/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 1/3 | Epoch 6/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 1/3 | Epoch 7/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 1/3 | Epoch 8/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 1/3 | Epoch 9/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 1/3 | Epoch 10/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 2/3 | Epoch 1/10 | Train Loss: 0.0004 | Val Loss: 0.0001
  Fold 2/3 | Epoch 2/10 | Train Loss: 0.0001 | Val Loss: 0.0001
  Fold 2/3 | Epoch 3/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 2/3 | Epoch 4/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 2/3 | Epoch 5/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 2/3 | Epoch 6/1

## GRU

In [21]:
class GRU(nn.Module):
    hidden_size = 64
    num_layers  = 2
    epochs      = 10
    batch_size  = 32
    lr          = 1e-3
 
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(INPUT_SIZE, self.hidden_size, self.num_layers, batch_first=True)
        self.fc  = nn.Linear(self.hidden_size, OUTPUT_SIZE)
 
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

In [22]:
gru_model, gru_pipeline = train(GRU, "GRU", experiments, train_keys, val_keys)
X_test_gru, y_test_gru    = experiments_to_tensor(test_keys, experiments, gru_pipeline, fit=False)
 
plot_validation_overview(gru_model, "GRU", test_keys, experiments, gru_pipeline)
plot_and_save_per_experiment(gru_model, "GRU", test_keys, experiments, gru_pipeline)


 Training GRU 
  Fold 1/3 | Epoch 1/10 | Train Loss: 0.0005 | Val Loss: 0.0001
  Fold 1/3 | Epoch 2/10 | Train Loss: 0.0001 | Val Loss: 0.0001
  Fold 1/3 | Epoch 3/10 | Train Loss: 0.0001 | Val Loss: 0.0001
  Fold 1/3 | Epoch 4/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 1/3 | Epoch 5/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 1/3 | Epoch 6/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 1/3 | Epoch 7/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 1/3 | Epoch 8/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 1/3 | Epoch 9/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 1/3 | Epoch 10/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 2/3 | Epoch 1/10 | Train Loss: 0.0004 | Val Loss: 0.0002
  Fold 2/3 | Epoch 2/10 | Train Loss: 0.0001 | Val Loss: 0.0001
  Fold 2/3 | Epoch 3/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 2/3 | Epoch 4/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 2/3 | Epoch 5/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 2/3 | Epoch 6/10

## RNN

In [23]:
class RNN(nn.Module):
    hidden_size = 64
    num_layers  = 2
    epochs      = 10
    batch_size  = 32
    lr          = 1e-3
 
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(INPUT_SIZE, self.hidden_size, self.num_layers, batch_first=True)
        self.fc  = nn.Linear(self.hidden_size, OUTPUT_SIZE)
 
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

In [24]:
rnn_model, rnn_pipeline = train(RNN, "RNN", experiments, train_keys, val_keys)
X_test_rnn, y_test_rnn    = experiments_to_tensor(test_keys, experiments, rnn_pipeline, fit=False)
 
plot_validation_overview(rnn_model, "RNN", test_keys, experiments, rnn_pipeline)
plot_and_save_per_experiment(rnn_model, "RNN", test_keys, experiments, rnn_pipeline)


 Training RNN 
  Fold 1/3 | Epoch 1/10 | Train Loss: 0.0006 | Val Loss: 0.0001
  Fold 1/3 | Epoch 2/10 | Train Loss: 0.0001 | Val Loss: 0.0001
  Fold 1/3 | Epoch 3/10 | Train Loss: 0.0001 | Val Loss: 0.0000
  Fold 1/3 | Epoch 4/10 | Train Loss: 0.0001 | Val Loss: 0.0001
  Fold 1/3 | Epoch 5/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 1/3 | Epoch 6/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 1/3 | Epoch 7/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 1/3 | Epoch 8/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 1/3 | Epoch 9/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 1/3 | Epoch 10/10 | Train Loss: 0.0000 | Val Loss: 0.0001
  Fold 2/3 | Epoch 1/10 | Train Loss: 0.0008 | Val Loss: 0.0002
  Fold 2/3 | Epoch 2/10 | Train Loss: 0.0002 | Val Loss: 0.0001
  Fold 2/3 | Epoch 3/10 | Train Loss: 0.0001 | Val Loss: 0.0001
  Fold 2/3 | Epoch 4/10 | Train Loss: 0.0001 | Val Loss: 0.0001
  Fold 2/3 | Epoch 5/10 | Train Loss: 0.0001 | Val Loss: 0.0001
  Fold 2/3 | Epoch 6/10

# **Evaluation**

In [27]:
def evaluate(model, model_name, X_test, y_test, pipeline):
    model.eval()
    with torch.no_grad():
        preds = model(X_test).squeeze().numpy()
    actuals = y_test.squeeze().numpy()

    n_cols = len(INPUT) + 1

    dummy_preds = np.zeros((len(preds), n_cols))
    dummy_preds[:, -1] = preds
    preds_orig = pipeline.scaler.inverse_transform(dummy_preds)[:, -1]

    dummy_actuals = np.zeros((len(actuals), n_cols))
    dummy_actuals[:, -1] = actuals
    actuals_orig = pipeline.scaler.inverse_transform(dummy_actuals)[:, -1]

    mae = mean_absolute_error(actuals_orig, preds_orig)
    mse = mean_squared_error(actuals_orig, preds_orig)
    rmse = np.sqrt(mse)
    std = np.std(actuals_orig)

    print(f"\n{model_name}")
    print(f"MAE  : {mae:.4f} ({mae/std*100:.2f}% of std)")
    print(f"MSE  : {mse:.4f}")
    print(f"RMSE : {rmse:.4f} ({rmse/std*100:.2f}% of std)")
    print(f"STD  : {std:.4f}")


evaluate(lstm_model, "LSTM", X_test_lstm, y_test_lstm, lstm_pipeline)
evaluate(gru_model, "GRU", X_test_gru, y_test_gru, gru_pipeline)
evaluate(rnn_model, "RNN", X_test_rnn, y_test_rnn, rnn_pipeline)


LSTM
MAE  : 9.2552 (4.07% of std)
MSE  : 289.1459
RMSE : 17.0043 (7.48% of std)
STD  : 227.2434

GRU
MAE  : 14.8957 (6.55% of std)
MSE  : 495.3049
RMSE : 22.2554 (9.79% of std)
STD  : 227.2434

RNN
MAE  : 10.4208 (4.59% of std)
MSE  : 297.9873
RMSE : 17.2623 (7.60% of std)
STD  : 227.2434
